In [1]:
from google import genai
from google.colab import userdata
import numpy as np
from google.colab import drive
import os

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

drive.mount('/content/drive')

folder_path = "/content/drive/MyDrive/RAG_Assignment"

file_path = os.path.join(folder_path, "saudi_cities.txt")

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

# Chunks
chunk_size = 500
overlap = 100

def create_chunks(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = create_chunks(text, chunk_size=500, overlap=100)


# Embedding
EMBEDDING_MODEL = "gemini-embedding-001"

def get_embedding(text):
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text
    )

    return np.array(response.embeddings[0].values)


chunk_embeddings = []

for chunk in chunks:
    embedding = get_embedding(chunk)
    chunk_embeddings.append(embedding)


query = input("Enter your question: ")

query_embedding = get_embedding(query)


# Cosine Similarity
def cosine_similarity(vector_a, vector_b):
    return np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    )


similarities = []

for i, embedding in enumerate(chunk_embeddings):
    score = cosine_similarity(query_embedding, embedding)

    similarities.append({
        "chunk_index": i,
        "score": score,
        "text": chunks[i]
    })


# Rank results
ranked_results = sorted(
    similarities,
    key=lambda x: x["score"],
    reverse=True
)

top_3 = ranked_results[:3]

print(f"Number of chunks: {len(chunks)}")
print(f"Number of embeddings: {len(chunk_embeddings)}")

# Display results
print("\nQuestion:")
print(query)

print("\nTop 3 Answers:\n")

for rank, result in enumerate(top_3, start=1):
    print(f"Answer {rank}")
    print(f"Similarity Score: {result['score']:.4f}")
    print(result["text"])
    print("-" * 80)

Mounted at /content/drive
Enter your question: what is the capital city of saudi arabia?
Number of chunks: 6
Number of embeddings: 6

Question:
what is the capital city of saudi arabia?

Top 3 Answers:

Answer 1
Similarity Score: 0.7615
Riyadh is the capital city of Saudi Arabia. It is one of the largest cities in the country and is located in the center of Saudi Arabia. Riyadh is an important center for business, government, and education.

Jeddah is a large city located on the Red Sea. It is known for its coastal location and its connection to the sea. Jeddah is also an important commercial city and has many shopping centers, restaurants, and hotels.

Makkah is one of the most important cities in Saudi Arabia. It is home to t
--------------------------------------------------------------------------------
Answer 2
Similarity Score: 0.6790
estaurants, and hotels.

Makkah is one of the most important cities in Saudi Arabia. It is home to the Kaaba, which is located inside the Grand Mos